# Forecast availability and recovery scenarios

Independent recent-rate extrapolation is a scenario, not a resource-leveled critical-path plan. Stalled incomplete packages prevent an overall finite completion date.

In [1]:
from pathlib import Path
import os, sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))


In [2]:
import pandas as pd
from src.forecast import forecast_packages
d=pd.read_csv('data/processed/ledger.csv')
f=forecast_packages(d,30)
stalled=f[(f.actual_quantity<f.scope_quantity)&f.current_finish.isna()]
display(f)
print('Stalled packages:',stalled.wbs.tolist())
print('Whole project:', 'UNAVAILABLE' if len(stalled) else f.current_finish.max())

,wbs,section,discipline,baseline_finish,scope_quantity,actual_quantity,weekly_rate,current_finish,recovery_finish,current_delay_days
0,S1-1,S1,Track,2026-10-12,18,13.339582,0.434932,2026-10-17 00:10:00.375917116,2026-10-02 00:08:00.300733693,5.006949
1,S1-2,S1,Electrification,2026-11-30,18,9.714396,0.444642,2026-12-11 10:34:04.372955728,2026-11-15 08:27:15.498364579,11.440328
2,S1-3,S1,Stations,2026-11-09,2,1.000000,0.000000,NaT,NaT,NaN
3,S1-4,S1,Signaling,2027-01-04,4,1.000000,0.000000,NaT,NaT,NaN
4,S2-1,S2,Track,2026-10-12,16,11.278707,0.383744,2026-10-28 02:56:42.182145202,2026-10-10 21:33:21.745716160,16.122710
5,S2-2,S2,Electrification,2026-11-30,16,8.251835,0.367024,2026-12-28 18:36:52.172349315,2026-11-29 05:17:29.737879454,28.775604
6,S2-3,S2,Stations,2026-11-09,2,1.000000,0.000000,NaT,NaT,NaN
7,S2-4,S2,Signaling,2027-01-04,4,1.000000,0.000000,NaT,NaT,NaN
8,S3-1,S3,Track,2026-10-12,20,11.036731,0.351161,2027-01-28 16:08:43.637411964,2026-12-23 22:30:58.909929575,108.672727
9,S3-2,S3,Electrification,2026-11-30,20,8.033079,0.361219,2027-03-22 21:42:38.489901661,2027-02-04 12:34:06.791921330,112.904612


Stalled packages: ['S1-3', 'S1-4', 'S2-3', 'S2-4', 'S3-3', 'S4-3', 'S4-4']
Whole project: UNAVAILABLE


In [3]:
scenarios=[]
for factor in [1.,1.1,1.25,1.4]:
    result=forecast_packages(d,30,factor)
    scenarios.append({'assumed_multiplier':factor,'latest_forecastable_recovery':result.recovery_finish.max(),'unforecastable_packages':int(result.current_finish.isna().sum())})
display(pd.DataFrame(scenarios))
print('Rates do not establish recovery feasibility. Validate resources, possession access and assurance first.')

,assumed_multiplier,latest_forecastable_recovery,unforecastable_packages
0,1.00,2027-03-22 21:42:38.489901661,7
1,1.10,2027-03-01 19:44:13.172637871,7
2,1.25,2027-02-04 12:34:06.791921330,7
3,1.40,2027-01-15 15:30:27.492786898,7


Rates do not establish recovery feasibility. Validate resources, possession access and assurance first.


In [4]:
from src.forecast import progress_scenarios
trajectories=progress_scenarios(d,30)
section=trajectories.groupby(['scenario','section','week'],as_index=False).agg(projected_ev=('projected_ev','sum'),budget=('budget','sum'))
section['progress']=section.projected_ev/section.budget
display(section[section.week.isin([30,40,52,65])])
print('Projected progress is DERIVED from SIMULATED quantities; rates are ASSUMPTIONS.')

,scenario,section,week,projected_ev,budget,progress
0,BASELINE,S1,30,5.161247e+07,80600000.0,0.640353
10,BASELINE,S1,40,7.205156e+07,80600000.0,0.893940
22,BASELINE,S1,52,8.060000e+07,80600000.0,1.000000
35,BASELINE,S1,65,8.060000e+07,80600000.0,1.000000
53,BASELINE,S2,30,4.797145e+07,75200000.0,0.637918
63,BASELINE,S2,40,6.704643e+07,75200000.0,0.891575
75,BASELINE,S2,52,7.520000e+07,75200000.0,1.000000
88,BASELINE,S2,65,7.520000e+07,75200000.0,1.000000
106,BASELINE,S3,30,5.525350e+07,86000000.0,0.642483
116,BASELINE,S3,40,7.705669e+07,86000000.0,0.896008


Projected progress is DERIVED from SIMULATED quantities; rates are ASSUMPTIONS.
